# Nemotron-3-Nano-30B — v0.9 SFT Training on Kaggle RTX Pro 6000

**Approach:** Format 4 SFT from base model, all 16 categories, incremental run support  
**Hardware target:** NVIDIA RTX Pro 6000 (96 GB VRAM, Blackwell SM 12.0)

## Prerequisites — add these inputs before running

| Input type | What to add | Why |
|---|---|---|
| **Model** | `metric/nemotron-3-nano-30b-a3b-bf16` (via Models tab) | Base model (~60 GB) |
| **Dataset** | `gdataranger/nemotron-v09-training-data` | Training data (`v0.9_train.jsonl`) |
| **Utility Script** | `gdataranger/nemotron-v09-build` | torch nightly cu128, mamba-ssm, trl, unsloth |

Right panel → **Accelerator → GPU (RTX Pro 6000)** → **Save Version → Save & Run All**.  
(RTX Pro 6000 must be selected manually — `enable_gpu: true` defaults to P100.)

## Incremental run strategy (9-hour limit)

Data stats: p50=3,457 tok / p90=6,831 tok / total=13,730 train examples.  
Model load takes ~30 min, leaving ~8.5 h for training (~680 steps at 45 s/step).

| Run | `WARMSTART_ADAPTER` | `MAX_SEQ_LENGTH` | `MIN_SEQ_LENGTH` | Examples | Steps | Est. time | Goal |
|---|---|---|---|---|---|---|---|
| **run6** | `None` (fresh) | `4096` | `0` | ~7,900 (58%) | ~495 | ~6.2 h | All short+medium examples, all modules incl. Mamba |
| **run7** | path to run6 | `7680` | `4096` | ~5,800 (42%) | ~363 | ~4.5 h | Long examples only, warmstart from run6 |

**Why 4096 not 2048?** The original plan's 2048 cutoff covers only 30% of training data (3.2 h).  
4096 covers 58% in one session. run6→run7 replaces the old run1→run2→run3 three-session chain.

**Why no warmstart from prior runs?** `PeftModel.from_pretrained` restores only modules  
listed in the saved adapter_config. An adapter trained without `in_proj`/`out_proj` would  
silently leave those 23 Mamba SSM layers frozen. Always start from `None` for the first run  
of a new target set.

After each run: the adapter is zipped automatically. Download, upload as a Kaggle dataset,  
then set `WARMSTART_ADAPTER` for the next session.

In [ ]:
import os

# ── RUN IDENTITY ──────────────────────────────────────────────────────────────
RUN_NAME   = "v9_run6"                  # run6=short fresh start; run7=long warmstart
OUTPUT_DIR = f"/kaggle/working/adapter_{RUN_NAME}"

# ── INCREMENTAL TRAINING ──────────────────────────────────────────────────────
# run6: fresh start (None).  run7: "/kaggle/input/nemotron-v9-run6/adapter_v9_run6"
# WARNING: only warmstart from an adapter trained with the same target_modules.
# Adapters without in_proj/out_proj will silently leave 23 Mamba layers frozen.
WARMSTART_ADAPTER = None

# ── DATA SLICE ────────────────────────────────────────────────────────────────
MAX_TRAIN_RECORDS = None   # None = use all examples within seq_len window
MAX_SEQ_LENGTH    = 4096   # run6: 4096 (58% of data, ~6.2 h); run7: 7680
MIN_SEQ_LENGTH    = 0      # run6: 0; run7: 4096 (skip examples already in run6)

# ── TRAINING HYPERPARAMETERS ──────────────────────────────────────────────────
MAX_STEPS     = None       # None = 1 full epoch (~495 steps at seq=4096)
LEARNING_RATE = 2e-4
BATCH_SIZE    = 1
GRAD_ACCUM    = 16          # effective batch = 16
LORA_R        = 32
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.0
SEED          = 3407

# ── ENVIRONMENT ───────────────────────────────────────────────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:512"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"
os.environ["HF_HOME"]                  = "/kaggle/working/.cache/huggingface"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

MODEL_HF_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

print(f"RUN_NAME:          {RUN_NAME}")
print(f"OUTPUT_DIR:        {OUTPUT_DIR}")
print(f"WARMSTART_ADAPTER: {WARMSTART_ADAPTER}")
print(f"MAX_TRAIN_RECORDS: {MAX_TRAIN_RECORDS}")
print(f"MAX_SEQ_LENGTH:    {MAX_SEQ_LENGTH}")
print(f"MIN_SEQ_LENGTH:    {MIN_SEQ_LENGTH}")
print(f"MAX_STEPS:         {MAX_STEPS}")

In [ ]:
import importlib.metadata, importlib.util, importlib, site, pathlib, subprocess, sys
from packaging.version import Version

# ── helpers ──────────────────────────────────────────────────────────────────
def _installed_ver(name):
    try:
        return Version(importlib.metadata.version(name))
    except Exception:
        pass
    # importlib.metadata may not rescan sys.path entries added mid-run.
    # Fall back to a direct dist-info scan across all sys.path entries.
    pkg_key = name.lower().replace('-', '_').replace('.', '_')
    for path_str in sys.path:
        p = pathlib.Path(path_str)
        if not p.is_dir():
            continue
        for dist_dir in p.glob('*.dist-info'):
            dist_name = dist_dir.name.split('-')[0].lower().replace('-', '_').replace('.', '_')
            if dist_name == pkg_key:
                try:
                    for line in (dist_dir / 'METADATA').read_text(errors='replace').splitlines():
                        if line.startswith('Version:'):
                            return Version(line.split(':', 1)[1].strip())
                except Exception:
                    pass
    return None

def _pip(pkgs, extra_args=None, label=None):
    cmd = [sys.executable, "-m", "pip", "install"] + (extra_args or []) + pkgs
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        tag = label or pkgs[0]
        no_net = ("Temporary failure in name resolution" in r.stderr or
                  "NewConnectionError" in r.stderr or
                  "Failed to establish a new connection" in r.stderr)
        if no_net:
            raise RuntimeError(
                "\n" + "="*60 + "\n"
                "pip: INTERNET IS OFF and packages not in build script output.\n"
                "  Ensure gdataranger/nemotron-v09-build is attached as a utility script.\n"
                + "="*60
            )
        print(f"\n{'='*60}")
        print(f"pip FAILED [{tag}] exit={r.returncode}")
        print("--- stderr ---")
        print(r.stderr if r.stderr.strip() else "(empty)")
        print(f"{'='*60}\n")
        raise RuntimeError("pip install failed — see output above")

def _load_build_script(candidates, label):
    """Add build script's python_packages dirs to sys.path. Returns root path or None."""
    root = next((pathlib.Path(p) for p in candidates if pathlib.Path(p).exists()), None)
    if root:
        pkg_dirs = sorted(root.rglob("python_packages"))
        for pkg_dir in pkg_dirs:
            if str(pkg_dir) not in sys.path:
                sys.path.insert(0, str(pkg_dir))
            site.addsitedir(str(pkg_dir))
        importlib.invalidate_caches()
        if not pkg_dirs:
            print(f"{label}: found at {root} but no python_packages — build may have failed")
        else:
            print(f"{label}: loaded {len(pkg_dirs)} python_packages dir(s) from {root}")
        return root
    for nb_root in [pathlib.Path("/kaggle/usr/lib/notebooks"), pathlib.Path("/kaggle/usr/lib")]:
        if nb_root.exists():
            print(f"{label}: not found. Contents of {nb_root}:")
            for p in sorted(nb_root.rglob("*"))[:30]:
                print(f"  {p}")
            break
    else:
        print(f"{label}: not found — attach gdataranger/nemotron-v09-build as a utility script")
    return None

# ── 1. gdataranger/nemotron-v09-build ────────────────────────────────────────
# MUST run before any `import torch` — mamba_ssm was compiled against nightly
# torch and will fail to load if system torch is imported first.
_util_root = _load_build_script([
    "/kaggle/usr/lib/notebooks/gdataranger/nemotron-v09-build",
    "/kaggle/usr/lib/notebooks/gdataranger/nemotron_v09_build",
], label="nemotron-v09-build")

# ── 2. Core ML packages ───────────────────────────────────────────────────────
_MIN_VERSIONS = {
    "transformers":     "4.40.0",
    "datasets":         "2.18.0",
    "accelerate":       "0.30.0",
    "peft":             "0.10.0",
    "trl":              "0.8.0",
    "safetensors":      "0.4.0",
    "scipy":            "1.10.0",
    "scikit-learn":     "1.3.0",
    "huggingface_hub":  "1.5.0",
}
_to_install = []
print("\nChecking package versions:")
for pkg, min_ver in _MIN_VERSIONS.items():
    cur = _installed_ver(pkg)
    ok = cur is not None and cur >= Version(min_ver)
    status = f"{cur} ✓" if ok else (f"{cur} → need >={min_ver}" if cur else "missing")
    print(f"  {pkg:20s} {status}")
    if not ok:
        _to_install.append(pkg)

if _to_install:
    print(f"\nInstalling from PyPI: {_to_install}")
    _pip([f"{p}>=0.8.0" if p == "trl" else p for p in _to_install], label="core")
    print("Core packages installed.")
else:
    print("All core packages satisfied.")

# ── 3. Unsloth ────────────────────────────────────────────────────────────────
if _installed_ver("unsloth") is None:
    print("\nInstalling Unsloth...")
    _pip(["unsloth", "unsloth_zoo"], extra_args=["--no-deps"], label="unsloth (PyPI)")
    print("Unsloth installed.")
else:
    print(f"\nunsloth: {_installed_ver('unsloth')} ✓")
# ── ptxas-blackwell: Kaggle util-script fs is read-only; copy to /tmp + chmod ─
import shutil as _shutil, os as _os, stat as _stat
_ptxas_src = _util_root and (_util_root / "python_packages/triton/backends/nvidia/bin/ptxas-blackwell")
if _ptxas_src and _ptxas_src.exists():
    _ptxas_dst = pathlib.Path("/tmp/ptxas-blackwell")
    _shutil.copy2(_ptxas_src, _ptxas_dst)
    _ptxas_dst.chmod(0o755)
    _os.environ["PTXAS_BLACKWELL"] = str(_ptxas_dst)
    print(f"ptxas-blackwell → {_ptxas_dst} (PTXAS_BLACKWELL set)")
else:
    print("ptxas-blackwell not found in build script — skipping")

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU not available — enable RTX Pro 6000 in notebook settings"

gpu_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory
free_vram, _ = torch.cuda.mem_get_info(0)
cc = torch.cuda.get_device_capability(0)

print(f"GPU:               {gpu_name}")
print(f"VRAM total:        {total_vram/1e9:.1f} GB")
print(f"VRAM free:         {free_vram/1e9:.1f} GB")
print(f"Compute cap:       SM {cc[0]}.{cc[1]}")
print(f"PyTorch:           {torch.__version__}")
print(f"CUDA runtime:      {torch.version.cuda}")

if total_vram < 80e9:
    print("WARNING: less than 80 GB VRAM — may OOM at seq_len=2048. Consider reducing MAX_SEQ_LENGTH.")
elif total_vram >= 90e9:
    print("Memory budget OK for seq_len=2048 training.")

In [ ]:
import importlib, importlib.util, sys, types
import unsloth  # must precede any transformers import

# Pre-insert a stub for selective_scan_cuda so mamba_ssm's legacy Mamba-1 path
# doesn't error on import. NemotronH uses Mamba-2 Triton kernels and never calls
# selective_scan_cuda at runtime, so a no-op stub is safe.
if 'selective_scan_cuda' not in sys.modules:
    sys.modules['selective_scan_cuda'] = types.ModuleType('selective_scan_cuda')

def _try_mamba():
    try:
        import mamba_ssm
        return True
    except Exception:
        return False

if _try_mamba():
    print("mamba_ssm: available")
else:
    _spec = importlib.util.find_spec("mamba_ssm")
    if _spec is not None:
        print(f"mamba_ssm found at {_spec.submodule_search_locations} but import failed.")
        print(f"  Current torch: {torch.__version__}  CUDA: {torch.version.cuda}")
        raise RuntimeError(
            "mamba_ssm import failed despite being in sys.path — possible torch ABI mismatch.\n"
            "Restart kernel and re-run (cell-install must run before any `import torch`)."
        )

    import urllib.request
    _has_internet = False
    try:
        urllib.request.urlopen('https://pypi.org', timeout=5)
        _has_internet = True
    except Exception:
        pass

    if not _has_internet:
        raise RuntimeError(
            "mamba_ssm not found in build script output and no internet available.\n"
            "Ensure gdataranger/nemotron-v09-build is attached and its last committed run succeeded."
        )

    import subprocess, os
    cc = torch.cuda.get_device_capability(0)
    arch_str = f"{cc[0]}.{cc[1]}+PTX"
    print(f"mamba_ssm not found — building from source for SM {arch_str} (~15 min)...")
    _env = {**os.environ, "TORCH_CUDA_ARCH_LIST": arch_str,
            "CAUSAL_CONV1D_FORCE_BUILD": "TRUE", "MAX_JOBS": "4"}
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "causal-conv1d", "--no-binary", "causal-conv1d", "--no-build-isolation"],
        env=_env, check=True
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "mamba-ssm", "--no-binary", "mamba-ssm", "--no-build-isolation"],
        env=_env, check=True
    )
    print(f"mamba_ssm available: {_try_mamba()}")

In [ ]:
import pathlib

_kaggle_model_candidates = [
    "/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",       # metric/... lowercase
    "/kaggle/input/nemotron-3-nano-30b-a3b-bf16/Transformers/default/1",       # metric/... uppercase T
    "/kaggle/input/nvidia-nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/nvidia-nemotron-3-nano-30b-a3b-bf16/Transformers/default/1",
    "/kaggle/input/nemotron3nano30b/transformers/default/1",
]
MODEL_PATH = next((p for p in _kaggle_model_candidates if pathlib.Path(p).exists()), None)

# Always dump /kaggle/input so we can diagnose mount issues
_input = pathlib.Path("/kaggle/input")
if MODEL_PATH is None:
    print("/kaggle/input contents:")
    for _p in sorted(_input.iterdir()):
        print(f"  {_p}")
        for _sub in sorted(_p.rglob("config.json"))[:2]:
            print(f"    config.json -> {_sub}")
            MODEL_PATH = str(_sub.parent)

if MODEL_PATH:
    print(f"Model (Kaggle local): {MODEL_PATH}")
else:
    MODEL_PATH = MODEL_HF_ID
    print(f"Kaggle model not found — downloading from HF Hub: {MODEL_PATH}")
    print("  (ensure internet is enabled in notebook settings)")


In [ ]:
import sys, warnings, torch

warnings.filterwarnings("ignore", message=r".*save_embedding_layers.*")
warnings.filterwarnings("ignore", message=r".*Could not find a config file.*")
warnings.filterwarnings("ignore", message=r".*Unable to fetch remote file.*")
warnings.filterwarnings("ignore", message=r".*use_return_dict.*deprecated.*")
warnings.filterwarnings("ignore", message=r".*unsloth_force_compile.*")
warnings.filterwarnings("ignore", message=r".*torchvision.*")

_unsloth_loaded = False
torch.cuda.empty_cache()

try:
    from unsloth import FastLanguageModel
    print(f"Loading model via FastLanguageModel (Unsloth): {MODEL_PATH}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        dtype=torch.bfloat16,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",  # required for NemotronH hybrid Mamba-2/attention
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    _unsloth_loaded = True
    print("FastLanguageModel loaded — MoE expert layers patched for LoRA")

except Exception as _e:
    import traceback
    print(f"FastLanguageModel failed — falling back to AutoModelForCausalLM")
    traceback.print_exc()
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map={"":0},
        low_cpu_mem_usage=True,
        attn_implementation="eager",
    )

torch.cuda.empty_cache()
free_gb, _ = torch.cuda.mem_get_info(0)
print(f"Model loaded (unsloth={_unsloth_loaded}). GPU free: {free_gb/1e9:.1f} GB")

In [ ]:
# Explicit LoRA targets — 'all-linear' triggers a Unsloth 2026.6.x discovery bug on NemotronH.
# Explicit names bypass the check and still hit all key projection layers.
# in_proj / out_proj cover the 23 Mamba SSM mixer layers (huikang reference uses both).
_LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                 "gate_proj", "up_proj", "down_proj", "in_proj", "out_proj"]
_lora_via_unsloth = False

if WARMSTART_ADAPTER:
    from peft import PeftModel
    print(f"Warmstart: loading adapter from {WARMSTART_ADAPTER}")
    model = PeftModel.from_pretrained(model, WARMSTART_ADAPTER, is_trainable=True)
    _lora_via_unsloth = _unsloth_loaded
    print("Adapter loaded and set to trainable (warmstart mode)")

elif _unsloth_loaded:
    try:
        model = FastLanguageModel.get_peft_model(
            model,
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=_LORA_TARGETS,
            use_gradient_checkpointing=False,  # managed manually in load cell
            random_state=SEED,
        )
        _lora_via_unsloth = True
        print("LoRA applied via FastLanguageModel.get_peft_model")
    except Exception as _e:
        import traceback
        print(f"FastLanguageModel.get_peft_model failed — falling back to standard PEFT")
        traceback.print_exc()

if not _lora_via_unsloth and not WARMSTART_ADAPTER:
    # Fallback: exclude gate/up/down (per-MoE-expert) to avoid 3+ GB adapter bloat.
    # in_proj / out_proj are per-Mamba-layer (23 total) — safe to include.
    _LORA_TARGETS_FALLBACK = ["q_proj", "k_proj", "v_proj", "o_proj", "in_proj", "out_proj"]
    from peft import LoraConfig, get_peft_model
    lora_cfg = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=_LORA_TARGETS_FALLBACK,
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)
    print(f"LoRA applied via standard PEFT (targets: {_LORA_TARGETS_FALLBACK})")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total_p:,} ({100*trainable/total_p:.2f}%)")

In [ ]:
import functools

# NemotronH native gradient checkpointing bypass.
# NemotronHForCausalLM sets supports_gradient_checkpointing=False which blocks the
# standard TRL/Unsloth enable path (silent no-op). Calling _set_gradient_checkpointing()
# directly walks GradientCheckpointingLayer subclasses and actually works.
# Without this: activation memory 20-40 GB at seq_len>=4096 -> OOM.
_gc_func = functools.partial(torch.utils.checkpoint.checkpoint, use_reentrant=False)
try:
    model.base_model.model._set_gradient_checkpointing(
        enable=True, gradient_checkpointing_func=_gc_func
    )
    model.enable_input_require_grads()
    print("Gradient checkpointing: enabled (NemotronH native, use_reentrant=False)")
except Exception as _e:
    print(f"Gradient checkpointing: unavailable ({_e})")

# Override Unsloth's conservative model.max_seq_length=2048 cap so SFTTrainer
# doesn't silently truncate MAX_SEQ_LENGTH to 2048.
_cap = getattr(model, "max_seq_length", None)
if _cap is not None and _cap < MAX_SEQ_LENGTH:
    model.max_seq_length = MAX_SEQ_LENGTH
    print(f"model.max_seq_length: {_cap} -> {MAX_SEQ_LENGTH}")

# Suppress use_return_dict FutureWarning (NemotronH reads it on every forward call).
_cfg = getattr(model, "config", None)
if _cfg is not None and "use_return_dict" not in _cfg.__dict__:
    _cfg.__dict__["use_return_dict"] = getattr(_cfg, "return_dict", True)
warnings.filterwarnings("ignore", category=FutureWarning, message=r".*use_return_dict.*")

# Mamba fast-path enable (no-op if module not loaded yet; will be hit during training)
def _patch_mamba_fastpath(m):
    for name, mod in sys.modules.items():
        if "modeling_nemotron_h" in name and hasattr(mod, "is_fast_path_available"):
            mod.is_fast_path_available = True
            print("Mamba fast path enabled")
            return
    print("Mamba fast path: module not yet loaded (will apply at first forward pass)")
_patch_mamba_fastpath(model)

In [ ]:
import json, random, pathlib
from collections import defaultdict
from datasets import Dataset

# Resolve training data — upload v0.9_train.jsonl as a Kaggle dataset and add it as input
_train_data_candidates = [
    "/kaggle/input/nemotron-v09-training-data/v0.9_train.jsonl",
    "/kaggle/input/nemotron-v09-training-data/nemotron-v09-training-data/v0.9_train.jsonl",
    "/kaggle/input/nemotron-training-data/v0.9_train.jsonl",
    "/kaggle/input/v09-training-data/v0.9_train.jsonl",
    "/kaggle/input/nemotron-v09/v0.9_train.jsonl",
]
TRAIN_DATA_PATH = next((p for p in _train_data_candidates if pathlib.Path(p).exists()), None)

# Glob fallback — search all /kaggle/input subdirs for the file
if TRAIN_DATA_PATH is None:
    _hits = sorted(pathlib.Path("/kaggle/input").rglob("v0.9_train.jsonl"))
    if _hits:
        TRAIN_DATA_PATH = str(_hits[0])
        print(f"Found via glob: {TRAIN_DATA_PATH}")
    else:
        print("/kaggle/input contents:")
        for _p in sorted(pathlib.Path("/kaggle/input").iterdir()):
            print(f"  {_p}")
            for _f in sorted(_p.rglob("*.jsonl"))[:3]:
                print(f"    {_f}")

if TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "v0.9_train.jsonl not found.\n"
        "Upload data/v0.9_train.jsonl as a Kaggle dataset and add it as notebook input.\n"
        f"Searched: {_train_data_candidates}"
    )
print(f"Training data: {TRAIN_DATA_PATH}")

# Load all records
random.seed(SEED)
all_records, all_labels = [], []
with open(TRAIN_DATA_PATH, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        all_records.append({"messages": r["messages"]})
        all_labels.append(r.get("category") or r.get("bucket", "other"))
print(f"Loaded {len(all_records)} total records, {len(set(all_labels))} categories")

# Subsample by record count (stratified by category)
if MAX_TRAIN_RECORDS is not None and MAX_TRAIN_RECORDS < len(all_records):
    by_cat = defaultdict(list)
    for rec, lbl in zip(all_records, all_labels):
        by_cat[lbl].append((rec, lbl))
    per_cat = max(1, MAX_TRAIN_RECORDS // len(by_cat))
    sampled_r, sampled_l = [], []
    for cat, items in sorted(by_cat.items()):
        random.shuffle(items)
        for rec, lbl in items[:per_cat]:
            sampled_r.append(rec)
            sampled_l.append(lbl)
    all_records, all_labels = sampled_r[:MAX_TRAIN_RECORDS], sampled_l[:MAX_TRAIN_RECORDS]
    print(f"Subsampled to {len(all_records)} records (max {MAX_TRAIN_RECORDS})")

# Filter by sequence length
print(f"Filtering: keep examples where {MIN_SEQ_LENGTH} < len <= {MAX_SEQ_LENGTH} tokens...")
kept_r, kept_l, n_short, n_long = [], [], 0, 0
for r, lbl in zip(all_records, all_labels):
    try:
        text = tokenizer.apply_chat_template(
            r["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=True
        )
    except TypeError:
        text = tokenizer.apply_chat_template(
            r["messages"], tokenize=False, add_generation_prompt=False
        )
    n_tok = len(tokenizer(text, truncation=False, add_special_tokens=False)["input_ids"])
    if n_tok <= MIN_SEQ_LENGTH:
        n_short += 1
    elif n_tok > MAX_SEQ_LENGTH:
        n_long += 1
    else:
        kept_r.append(r)
        kept_l.append(lbl)

records, strat_labels = kept_r, kept_l
print(f"Kept: {len(records)} | Skipped short (<=MIN): {n_short} | Dropped long (>MAX): {n_long}")

dataset = Dataset.from_list(records)
print(f"Training dataset: {len(dataset)} examples across {len(set(strat_labels))} categories")

In [ ]:
import math
from torch.utils.data import DataLoader, Sampler
from trl import SFTConfig, SFTTrainer

def formatting_func(example):
    msgs = example["messages"]
    conversations = [msgs] if msgs and isinstance(msgs[0], dict) else msgs
    texts = []
    for conv in conversations:
        try:
            text = tokenizer.apply_chat_template(
                conv, tokenize=False, add_generation_prompt=False, enable_thinking=True
            )
        except TypeError:
            text = tokenizer.apply_chat_template(
                conv, tokenize=False, add_generation_prompt=False
            )
        texts.append(text)
    return texts

def build_stratified_order(labels, batch_size, seed):
    by_label = defaultdict(list)
    for idx, label in enumerate(labels):
        by_label[label].append(idx)
    rng = random.Random(seed)
    for v in by_label.values():
        rng.shuffle(v)
    n_batches = max(1, math.ceil(len(labels) / batch_size))
    batches = [[] for _ in range(n_batches)]
    order = list(range(n_batches))
    rng.shuffle(order)
    i = 0
    for label in sorted(by_label.keys()):
        for idx in by_label[label]:
            batches[order[i % n_batches]].append(idx)
            i += 1
    return [idx for b in batches for idx in b]

class OrderedSampler(Sampler):
    def __init__(self, order):
        self.order = list(order)
    def __iter__(self):
        return iter(self.order)
    def __len__(self):
        return len(self.order)

class StratifiedSFTTrainer(SFTTrainer):
    def __init__(self, *a, stratified_order=None, **kw):
        super().__init__(*a, **kw)
        self._stratified_order = stratified_order

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        try:
            return super().compute_loss(model, inputs, return_outputs, num_items_in_batch)
        except RuntimeError as err:
            if "view and is being modified inplace" not in str(err):
                raise
            # UnslothFusedLossBackward returns a view tensor that triggers autograd
            # inplace constraint in single-GPU context. Fall back to base Trainer path.
            from transformers import Trainer as _BaseTrainer
            print("Warning: UnslothFusedLossBackward inplace fallback", flush=True)
            return _BaseTrainer.compute_loss(self, model, inputs, return_outputs, num_items_in_batch)

    def get_train_dataloader(self):
        if self._stratified_order is None:
            return super().get_train_dataloader()
        keep = [c for c in self.train_dataset.column_names
                if c in ("input_ids", "attention_mask", "labels")]
        ds = self.train_dataset.select_columns(keep)
        return DataLoader(
            ds,
            batch_size=self.args.per_device_train_batch_size,
            sampler=OrderedSampler(self._stratified_order),
            collate_fn=self.data_collator,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
        )

In [ ]:
eff_bs   = BATCH_SIZE * GRAD_ACCUM
n_steps  = MAX_STEPS if MAX_STEPS is not None else max(1, len(records) // eff_bs)
warmup   = min(50, max(1, n_steps // 10))
strat_order = build_stratified_order(strat_labels, eff_bs, SEED)

print(f"Effective batch size : {eff_bs}")
print(f"Training steps       : {n_steps}")
print(f"Warmup steps         : {warmup}")
print(f"Examples processed   : ~{n_steps * eff_bs} (>{len(records)} = multiple passes)")

import pathlib
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_steps=n_steps,
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="linear",
    warmup_steps=warmup,
    adam_beta1=0.9,
    adam_beta2=0.95,
    adam_epsilon=1e-8,
    weight_decay=0.0,
    max_grad_norm=1e9,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=False,       # disabled here; enabled manually above via NemotronH native path
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2,
    remove_unused_columns=False,
    seed=SEED,
    report_to="none",
    packing=False,
)
# Unsloth sets padding_free=True; it rejects max_length != None without packing.
# Sequences are already filtered to MAX_SEQ_LENGTH in the data cell — safe to clear.
for _attr in ("max_seq_length", "max_length"):
    if getattr(training_args, _attr, None) is not None:
        setattr(training_args, _attr, None)

# Unsloth's new_init replaces training_args with a fresh config built from
# model.max_seq_length — clear it here so args.max_length stays None.
# Sequences are already filtered to MAX_SEQ_LENGTH in the data cell.
if hasattr(model, "max_seq_length"):
    model.max_seq_length = None

trainer = StratifiedSFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    formatting_func=formatting_func,
    stratified_order=strat_order,
)
print("Trainer ready.")

In [ ]:
import time

print("Starting training...")
t0 = time.time()
trainer.train()
elapsed = time.time() - t0

print(f"\nTraining complete: {elapsed/60:.1f} min ({elapsed:.0f} s)")
torch.cuda.empty_cache()
free_gb, _ = torch.cuda.mem_get_info(0)
print(f"GPU free after training: {free_gb/1e9:.1f} GB")
print(f"Avg time/step: {elapsed/n_steps:.1f} s")

In [ ]:
import pathlib
from safetensors.torch import load_file

pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

st_path = pathlib.Path(OUTPUT_DIR) / "adapter_model.safetensors"
if st_path.exists():
    saved = load_file(str(st_path))
    print(f"Adapter keys saved: {len(saved)}")
    print("  (expect ~510 keys: 418 MoE+attention + 92 Mamba in_proj/out_proj × 23 layers)")

print(f"\nFiles in {OUTPUT_DIR}:")
total_mb = 0
for f in sorted(pathlib.Path(OUTPUT_DIR).iterdir()):
    if f.is_file():
        mb = f.stat().st_size / 1024 / 1024
        total_mb += mb
        print(f"  {f.name}: {mb:.1f} MB")
print(f"Total: {total_mb:.1f} MB")

print(f"""
\n=== NEXT STEPS (run6 → run7) ===
1. Download adapter_v9_run6.zip from the Kaggle output panel
2. Upload as a Kaggle dataset: gdataranger/nemotron-v9-run6
3. For run7 (long examples), set in cell-config:
     RUN_NAME          = "v9_run7"
     WARMSTART_ADAPTER = "/kaggle/input/nemotron-v9-run6/adapter_v9_run6"
     MAX_SEQ_LENGTH    = 7680
     MIN_SEQ_LENGTH    = 4096   # skip examples already trained in run6
     MAX_STEPS         = None   # ~363 steps, ~4.5 h
""")

In [ ]:
import json, pathlib, shutil, subprocess, zipfile

ZIP_PATH = pathlib.Path(f"/kaggle/working/adapter_{RUN_NAME}.zip")
ADAPTER_DIR = pathlib.Path(OUTPUT_DIR)

# ── 1. Zip adapter ────────────────────────────────────────────────────────────
print(f"Zipping {ADAPTER_DIR.name} ...")
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(ADAPTER_DIR.rglob('*')):
        if f.is_file():
            zf.write(f, f.relative_to(ADAPTER_DIR.parent))
zip_mb = ZIP_PATH.stat().st_size / 1e6
print(f"Zip: {ZIP_PATH.name}  ({zip_mb:.1f} MB)")
print(f"Download from Kaggle output panel → {ZIP_PATH.name}")

# ── 2. Best-effort dataset push (requires internet; skipped on RTX Pro 6000) ──
# On success the adapter appears at kaggle.com/datasets/gdataranger/<slug>
# and can be added as a dataset source for the next training run.
_SLUG = f"gdataranger/nemotron-v09-adapter-{RUN_NAME.replace('_', '-')}"
_staging = pathlib.Path(f"/tmp/adapter-dataset-{RUN_NAME}")
_staging.mkdir(exist_ok=True)
shutil.copy2(ZIP_PATH, _staging / ZIP_PATH.name)
(_staging / "dataset-metadata.json").write_text(json.dumps({
    "title": f"Nemotron v09 Adapter {RUN_NAME}",
    "id": _SLUG,
    "licenses": [{"name": "CC0-1.0"}],
}, indent=2))

def _kaggle_push(staging, slug):
    r = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(staging), "--quiet"],
        capture_output=True, text=True, timeout=180,
    )
    if r.returncode == 0:
        print(f"Dataset created: https://www.kaggle.com/datasets/{slug}")
        return
    if "already exists" in r.stderr or "409" in r.stderr or "already exists" in r.stdout:
        r2 = subprocess.run(
            ["kaggle", "datasets", "version", "-p", str(staging),
             "-m", f"adapter {RUN_NAME}", "--quiet"],
            capture_output=True, text=True, timeout=180,
        )
        if r2.returncode == 0:
            print(f"Dataset updated: https://www.kaggle.com/datasets/{slug}")
            return
        print(f"Dataset version push failed: {(r2.stderr or r2.stdout)[:300]}")
        return
    no_net = any(s in r.stderr for s in ("Name or service not known", "Temporary failure",
                                          "NewConnectionError", "Failed to establish"))
    if no_net:
        print("Dataset push skipped — no internet (RTX Pro 6000). Download the zip manually.")
    else:
        print(f"Dataset push failed: {(r.stderr or r.stdout)[:300]}")

try:
    _kaggle_push(_staging, _SLUG)
except Exception as _e:
    print(f"Dataset push skipped: {_e}")

## Session-to-Session Resume Guide

### After run6 completes — upload the adapter

```bash
# Adapter is auto-zipped to /kaggle/working/adapter_v9_run6.zip
# Download from Kaggle output panel, then upload as a dataset:
kaggle datasets create -p /tmp/adapter-run6 --dir-mode zip
```

### run7 configuration (long examples, warmstart from run6)

```python
RUN_NAME          = "v9_run7"
WARMSTART_ADAPTER = "/kaggle/input/nemotron-v9-run6/adapter_v9_run6"
MAX_TRAIN_RECORDS = None
MAX_SEQ_LENGTH    = 7680
MIN_SEQ_LENGTH    = 4096   # skip short examples already trained in run6
MAX_STEPS         = None   # ~363 steps, ~4.5 h
```

**Warmstart safety check**: run7's warmstart adapter (run6) was trained with `in_proj`/`out_proj`  
in `target_modules`. All 23 Mamba SSM layers will continue training correctly.

### Memory estimate (RTX Pro 6000, 96 GB)

| Phase | Model BF16 | LoRA + AdamW | Activations (GC) | Peak | Status |
|---|---|---|---|---|---|
| Model load | 60 GB | — | — | 60 GB | ✓ 36 GB free |
| seq=4096 train | 60 GB | ~2 GB | ~10 GB | ~72 GB | ✓ safe |
| seq=7680 train | 60 GB | ~2 GB | ~15 GB | ~77 GB | ✓ fits |

Gradient checkpointing is enabled via `_set_gradient_checkpointing()` (NemotronH native bypass).